In [7]:
# ============================================================
# WAVELET-ENHANCED FEEDFORWARD NEURAL NETWORK
# - same structure as the FFNN template
# - same region-by-region estimation
# - same nested CV tuning logic
# - same per-region pipeline save
# - no pywt required
# - uses analytical wavelet basis expansions (mexican_hat / morlet)
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import datetime
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import HalvingGridSearchCV, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [8]:


# ============================================================
# CONFIG
# ============================================================

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

INPUT_CSV = "../EDA/region_temp_extended.csv"
OUTPUT_DIR = Path("../Outputs/WaveletFNN")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

RANDOM_STATE = 23
N_JOBS = 1
HORIZON = 1

DATE_COL = "date"
REGION_CANDIDATES = ["name", "region_code"]
TARGET_COL = "next_day"

FEATURE_COLS = [
    "dayofyear",
    "pdtn_doy",
    "de_trend_seas",
    "dts_doyavge",
    "dts_doyvar",
    "yday",
    "IIdays_ago",
    "IIIdays_ago",
    "IVdays_ago",
    "Vdays_ago",
    "VIdays_ago",
    "VIIdays_ago",
    "last_year",
    "last_2year",
    "last_3year",
    "last_4year",
    "last_5year",
    "h1_last_year",
    "h1_last_2years",
    "h1_last_3years",
    "h1_last_4years",
    "h1_last_5years",
    "h1_ly_2days",
    "h1_ly_3day",
    "h1_ly_4day",
    "h1_ly_5day",
    "h1_ly_6day",
    "h1_ly_7day",
    "h1_ly_next_1day",
    "h1_ly_next_2days",
    "h1_ly_next_3days",
    "h1_ly_next_4days",
    "h1_ly_next_5days",
    "h1_ly_next_6days",
    "h1_ly_next_7days",
    "diff_1year",
    "diff_2year",
    "diff_3year",
    "diff_4year",
    "diff_5year",
    "diff_yday",
    "diff_2days",
    "diff_3days",
    "diff_4days",
    "diff_5days",
    "diff_6days",
    "diff_7days",
    "last_7_1day_deltas_mean",
    "last_7_1day_deltas_min",
    "last_7_1day_deltas_max",
]

WAVELET_FNN_GRID = {
    "wavelet__n_wavelets": [1, 2, 3, 5],
    "wavelet__wavelet": ["mexican_hat", "morlet"],
    "wavelet__include_original": [True],
    "mlp__hidden_layer_sizes": [(50,), (25,), (12,), (25, 10), (25, 10, 5)],
    "mlp__activation": ["relu", "tanh"],
    "mlp__learning_rate": ["constant", "invscaling", "adaptive"],
    "mlp__learning_rate_init": [0.001, 0.005],
    "mlp__max_iter": [2000],
    "mlp__tol": [0.0001, 0.0005],
    "mlp__early_stopping": [True],
    "mlp__validation_fraction": [0.1, 0.2],
}

OUTER_CV = TimeSeriesSplit(n_splits=5, test_size=365)
INNER_CV = TimeSeriesSplit(n_splits=3)


# ============================================================
# HELPERS
# ============================================================

def find_region_col(df: pd.DataFrame) -> str:
    for c in REGION_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find region column among {REGION_CANDIDATES}")


def create_next_day_target(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL)
    g[TARGET_COL] = g["de_trend_seas"].shift(-1)
    return g


def load_data() -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(INPUT_CSV)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    region_col = find_region_col(df)

    required = {DATE_COL, region_col, *FEATURE_COLS}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.sort_values([region_col, DATE_COL]).reset_index(drop=True)
    return df, region_col


def evaluate_fit(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mae,
        "MAPE": mape,
        "Bias": bias,
        "R2": r2,
        "RMSE": rmse,
    }


# ============================================================
# WAVELET FEATURE TRANSFORMER
# ============================================================

class WaveletFeatureTransformer(BaseEstimator, TransformerMixin):
    """
    Analytical wavelet-basis feature expansion.
    Applies Mexican Hat or Morlet basis functions across the sample axis
    for each input feature and returns the expanded feature matrix.
    """

    def __init__(self, n_wavelets=3, wavelet="mexican_hat", include_original=True):
        self.n_wavelets = n_wavelets
        self.wavelet = wavelet
        self.include_original = include_original

    def fit(self, X, y=None):
        X = self._to_numpy(X)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = self._to_numpy(X)

        n_samples, n_features = X.shape
        t = np.linspace(-1.0, 1.0, n_samples)

        features = []
        if self.include_original:
            features.append(X)

        for j in range(n_features):
            x = X[:, j]

            for i in range(self.n_wavelets):
                if self.n_wavelets == 1:
                    shift = 0.0
                else:
                    shift = -0.75 + i * (1.5 / (self.n_wavelets - 1))

                scale = 0.25 + 0.20 * i
                tau = (t - shift) / scale

                if self.wavelet == "mexican_hat":
                    psi = (1.0 - tau**2) * np.exp(-0.5 * tau**2)
                elif self.wavelet == "morlet":
                    psi = np.cos(5.0 * tau) * np.exp(-0.5 * tau**2)
                else:
                    raise ValueError(f"Unknown wavelet type: {self.wavelet}")

                features.append((x * psi).reshape(-1, 1))

        X_out = np.hstack(features).astype(float)

        if not np.isfinite(X_out).all():
            raise ValueError("WaveletFeatureTransformer produced NaN/inf values.")

        return X_out

    @staticmethod
    def _to_numpy(X):
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        return np.asarray(X, dtype=float)


# ============================================================
# PIPELINE
# ============================================================

def build_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer()),
            ("scaler_pre", StandardScaler()),
            ("wavelet", WaveletFeatureTransformer()),
            ("scaler_post", StandardScaler()),
            ("mlp", MLPRegressor(random_state=RANDOM_STATE)),
        ]
    )


# ============================================================
# MAIN
# ============================================================

def main():
    feat_df, region_col = load_data()

    feat_df = (
        feat_df.groupby(region_col, group_keys=False)
        .apply(create_next_day_target)
        .reset_index(drop=True)
    )

    nested_cv_results = []
    best_params_dict = {}
    best_params_rows = []
    fit_metric_rows = []

    for region in feat_df[region_col].dropna().unique():
        print(f"\n=== REGION {region} ===")

        region_df = (
            feat_df.loc[
                feat_df[region_col] == region,
                [DATE_COL, region_col] + FEATURE_COLS + [TARGET_COL]
            ]
            .copy()
            .sort_values(DATE_COL)
        )

        train = region_df.dropna(axis=0, how="any").copy()

        if len(train) < 365 * 6:
            print(f"Skipping {region}: too few rows ({len(train)})")
            continue

        X = train[FEATURE_COLS]
        y = train[TARGET_COL]

        fold_results = []

        for fold_idx, (train_idx, test_idx) in enumerate(OUTER_CV.split(X), start=1):
            print(f"Outer Fold {fold_idx}")

            X_outer_train = X.iloc[train_idx]
            X_outer_test = X.iloc[test_idx]
            y_outer_train = y.iloc[train_idx]
            y_outer_test = y.iloc[test_idx]

            pipe = build_pipeline()

            grid_search = HalvingGridSearchCV(
                estimator=pipe,
                param_grid=WAVELET_FNN_GRID,
                cv=INNER_CV,
                scoring="neg_mean_absolute_percentage_error",
                refit=True,
                n_jobs=N_JOBS,
                factor=2,
            )

            grid_search.fit(X_outer_train, y_outer_train)

            best_model = grid_search.best_estimator_
            y_pred = best_model.predict(X_outer_test)

            mape = mean_absolute_percentage_error(y_outer_test, y_pred)
            r2 = r2_score(y_outer_test, y_pred)

            row = {
                "model": "wavelet_fnn",
                "region": region,
                "mape": mape,
                "r2": r2,
                "horizon": HORIZON,
                "fold": fold_idx,
            }
            nested_cv_results.append(row)
            fold_results.append(row)

            fold_key = f"{region}_fold{fold_idx}"
            best_params_dict[fold_key] = {
                "model": "wavelet_fnn",
                "params": grid_search.best_params_,
            }

            fold_best_params = pd.DataFrame([grid_search.best_params_], index=[fold_key])
            fold_best_params["model"] = "wavelet_fnn"
            fold_best_params["region"] = region
            fold_best_params["fold"] = fold_idx
            best_params_rows.append(fold_best_params)

        nested_cv_df = pd.DataFrame(fold_results)
        if nested_cv_df.empty:
            continue

        min_row = nested_cv_df.loc[nested_cv_df["mape"].idxmin()]
        best_fold = f"{min_row['region']}_fold{int(min_row['fold'])}"
        best_params = best_params_dict[best_fold]["params"]

        final_pipe = build_pipeline()
        final_pipe.set_params(**best_params)
        final_pipe.fit(X, y)

        y_fit = final_pipe.predict(X)
        fit_metrics = evaluate_fit(y, y_fit)
        fit_metrics["Model"] = "wavelet_fnn"
        fit_metrics["region"] = region
        fit_metric_rows.append(fit_metrics)

        with open(OUTPUT_DIR / f"wavelet_fnn_pipeline_{region}.pkl", "wb") as f:
            pickle.dump(final_pipe, f)

    if best_params_rows:
        best_params_df = pd.concat(best_params_rows, axis=0)
        best_params_df.to_csv(OUTPUT_DIR / f"wavelet_fnn_best_params_{TODAY}.csv")

    if nested_cv_results:
        nested_cv_df_all = pd.DataFrame(nested_cv_results)
        nested_cv_df_all.to_csv(OUTPUT_DIR / f"wavelet_fnn_nested_cv_{TODAY}.csv", index=False)

    if fit_metric_rows:
        fit_metrics_df = pd.DataFrame(fit_metric_rows)
        fit_metrics_df.to_csv(OUTPUT_DIR / f"wavelet_fnn_fit_metrics_{TODAY}.csv", index=False)

    print("Time taken:", datetime.datetime.now() - START)


if __name__ == "__main__":
    main()


=== REGION 11 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 24 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 27 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 28 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 32 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 44 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 52 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 53 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5
Time taken: 7:56:06.361175
